In [16]:
import numpy as np
import pandas as pd
import torch



In [18]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [19]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [20]:
from sklearn.model_selection import train_test_split

In [22]:
X = df.drop('diagnosis', axis = 1)
y = df['diagnosis']

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
from sklearn.preprocessing import LabelEncoder

In [29]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [31]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [41]:
class MySimpleNN():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype = torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred
  def binary_cross_entropy_loss(self, y_pred, y):
    epsilon = 1e-8
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = -( (y_train_tensor)* torch.log(y_pred) + (1-y_train_tensor) *torch.log(1- y_pred)).mean()
    return loss

In [42]:
learning_rate = 0.1
epochs = 25

In [43]:
model = MySimpleNN(X_train_tensor)

In [45]:
for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor)
  loss = model.binary_cross_entropy_loss(y_pred, y_train_tensor)
  loss.backward()
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate* model.bias.grad
  model.weights.grad.zero_()
  model.bias.grad.zero_()

In [46]:
print(f'Epoch: {epoch + 1}, loss: {loss.item()}')

Epoch: 25, loss: 0.8016469230977938


In [47]:
model.weights

tensor([[ 0.3307],
        [-0.2194],
        [-0.0444],
        [-0.1068],
        [ 0.2965],
        [-0.0084],
        [ 0.3388],
        [-0.3659],
        [-0.1261],
        [ 0.5487],
        [ 0.1317],
        [ 0.3377],
        [-0.3629],
        [ 0.0650],
        [ 0.4735],
        [-0.3492],
        [ 0.0496],
        [-0.2068],
        [ 0.4022],
        [-0.2363],
        [ 0.1074],
        [ 0.1666],
        [ 0.4548],
        [ 0.1198],
        [ 0.3005],
        [-0.1237],
        [-0.2054],
        [-0.1109],
        [-0.2582],
        [-0.0762]], dtype=torch.float64, requires_grad=True)

In [48]:
model.bias

tensor([-0.1704], dtype=torch.float64, requires_grad=True)

In [49]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6077254414558411
